# Arrays — Storage, Operations & Sorting

An **array** stores $n$ elements in a contiguous block of memory, so the address of element $i$ is computed directly as $\text{base} + i \cdot s$ for element size $s$. This makes random access $O(1)$ but forces shifting on insertion and deletion, and it is the backdrop against which every sorting algorithm trades comparisons for movements. Each algorithm below records a snapshot per step so its execution can be replayed and *watched*, not just summarized.

$$ \text{addr}(i) = \text{base} + i \cdot s, \qquad \text{access} = O(1),\quad \text{insert/delete} = O(n). $$

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.figsize'] = (9, 4.5)

# Each algorithm records snapshot frames; a Play button + slider replays them.
def make_player(n_steps, render, label='step'):
    slider = widgets.IntSlider(value=0, min=0, max=n_steps-1, description=label,
                               continuous_update=False, layout=widgets.Layout(width='60%'))
    play = widgets.Play(value=0, min=0, max=n_steps-1, interval=350)
    widgets.jslink((play, 'value'), (slider, 'value'))
    out = widgets.interactive_output(render, {'k': slider})
    display(widgets.HBox([play, slider]), out)

## Random Access via Address Arithmetic

Because elements are equally spaced, index $i$ maps to a byte offset with no traversal — reaching position 0 costs the same as position $n-1$. The widget computes the offset and highlights the targeted cell directly.

$$ \text{offset}(i) = i \cdot s. $$

In [2]:
def show_access(index, elem_size):
    n = 12
    fig, ax = plt.subplots(figsize=(9, 2.4))
    for i in range(n):
        color = 'tomato' if i == index else 'lightsteelblue'
        ax.add_patch(plt.Rectangle((i, 0), 1, 1, facecolor=color, edgecolor='k'))
        ax.text(i + 0.5, 0.5, f'a[{i}]', ha='center', va='center', fontsize=8)
        ax.text(i + 0.5, -0.35, f'{i*elem_size}', ha='center', va='center', fontsize=7, color='gray')
    ax.set_xlim(0, n); ax.set_ylim(-0.7, 1.2)
    ax.set_title(f'offset(a[{index}]) = {index} x {elem_size} = {index*elem_size} bytes  (single step)')
    ax.axis('off'); plt.show()

idx_s  = widgets.IntSlider(value=5, min=0, max=11, description='index i')
size_s = widgets.Dropdown(options=[1, 4, 8], value=8, description='elem size s')
display(widgets.HBox([idx_s, size_s]),
        widgets.interactive_output(show_access, {'index': idx_s, 'elem_size': size_s}))

Output()

## Insertion Shifts Elements, One Move at a Time

Inserting at position $k$ shifts the $n-k$ trailing elements one slot right. Stepping through reveals that the cost *is* the sequence of individual moves — not an abstract count. Each frame performs exactly one shift, then the value lands in the freed slot.

$$ \text{moves}_{\text{insert}}(k) = n - k. $$

In [ ]:
def insert_steps(a, k, value):
    a = list(a); frames = [(list(a), None, 'start')]
    a.append(None)
    for j in range(len(a)-1, k, -1):       # shift right, one element per frame
        a[j] = a[j-1]
        frames.append((list(a), j, f'shift a[{j-1}] -> a[{j}]'))
    a[k] = value
    frames.append((list(a), k, f'place {value} at a[{k}]'))
    return frames

frames_ins = insert_steps([10,20,30,40,50,60,70], k=2, value=99)
def draw_insert(k):
    arr, hot, note = frames_ins[k]
    fig, ax = plt.subplots(figsize=(9, 2.6))
    for i, v in enumerate(arr):
        c = 'tomato' if i == hot else ('lightgray' if v is None else 'lightsteelblue')
        ax.add_patch(plt.Rectangle((i, 0), 1, 1, facecolor=c, edgecolor='k'))
        ax.text(i+0.5, 0.5, '.' if v is None else str(v), ha='center', va='center')
    ax.set_xlim(0, len(arr)); ax.set_ylim(0, 1)
    ax.set_title(f'step {k}/{len(frames_ins)-1}: {note}'); ax.axis('off'); plt.show()
make_player(len(frames_ins), draw_insert)

Output()

## Search: Watch Each Probe

Linear search inspects elements left to right; binary search repeatedly halves a $[\text{lo},\text{hi}]$ window (greyed cells are excluded). Stepping through both on the *same* sorted array makes the $O(n)$ vs. $O(\log n)$ difference tangible — count the frames each one needs.

$$ \text{probes}_{\text{binary}} \le \lfloor \log_2 n \rfloor + 1. $$

In [4]:
def linear_frames(a, target):
    frames = []
    for i in range(len(a)):
        frames.append((i, 0, len(a)-1, a[i] == target))
        if a[i] == target: break
    return frames

def binary_frames(a, target):
    lo, hi, frames = 0, len(a)-1, []
    while lo <= hi:
        mid = (lo+hi)//2
        hit = a[mid] == target
        frames.append((mid, lo, hi, hit))
        if hit: break
        if a[mid] < target: lo = mid+1
        else: hi = mid-1
    return frames

a_search = np.arange(0, 32, 2)   # sorted, 16 elements
mode_s = widgets.Dropdown(options=['linear', 'binary'], value='binary', description='method')
tgt_s  = widgets.Dropdown(options=[int(v) for v in a_search], value=int(a_search[-3]), description='target')
search_area = widgets.Output()

def relaunch_search(*_):
    frames = linear_frames(a_search, tgt_s.value) if mode_s.value=='linear' else binary_frames(a_search, tgt_s.value)
    def draw(k):
        probe, lo, hi, hit = frames[k]
        fig, ax = plt.subplots(figsize=(10, 2.6))
        for i, v in enumerate(a_search):
            c = 'lightgray' if not (lo <= i <= hi) else 'lightsteelblue'
            if i == probe: c = 'seagreen' if hit else 'tomato'
            ax.add_patch(plt.Rectangle((i, 0), 1, 1, facecolor=c, edgecolor='k'))
            ax.text(i+0.5, 0.5, str(v), ha='center', va='center', fontsize=8)
        msg = 'FOUND' if hit else f'probe a[{probe}]={a_search[probe]}'
        ax.set_xlim(0, len(a_search)); ax.set_ylim(0, 1)
        ax.set_title(f'{mode_s.value} step {k+1}/{len(frames)}: {msg}'); ax.axis('off'); plt.show()
    search_area.clear_output(wait=True)
    with search_area: make_player(len(frames), draw)

mode_s.observe(relaunch_search, 'value'); tgt_s.observe(relaunch_search, 'value')
display(widgets.HBox([mode_s, tgt_s]), search_area)
relaunch_search()

Output()

## Elementary Sorts: Watch the Array Reorder

Insertion, selection, and bubble sort are all $O(n^2)$ but move elements differently. Recording a snapshot after each comparison/swap lets you replay the reordering while the running comparison ($C$) and swap ($S$) counters climb — selection swaps rarely, insertion is gentle on near-sorted input, bubble swaps a lot.

$$ C_{\text{worst}} = \frac{n(n-1)}{2} = O(n^2). $$

In [5]:
def bubble_frames(a):
    a = list(a); f = [(list(a), -1, -1, 0, 0, 'start')]; C = S = 0; n = len(a)
    for i in range(n):
        for j in range(n-1-i):
            C += 1
            if a[j] > a[j+1]:
                a[j], a[j+1] = a[j+1], a[j]; S += 1
                f.append((list(a), j, j+1, C, S, f'swap {j},{j+1}'))
            else:
                f.append((list(a), j, j+1, C, S, f'compare {j},{j+1}'))
    f.append((list(a), -1, -1, C, S, 'sorted')); return f

def insertion_frames(a):
    a = list(a); f = [(list(a), -1, -1, 0, 0, 'start')]; C = S = 0
    for i in range(1, len(a)):
        key = a[i]; j = i-1
        while j >= 0:
            C += 1
            if a[j] > key:
                a[j+1] = a[j]; S += 1; j -= 1
                f.append((list(a), j+1, i, C, S, f'shift into {j+2}'))
            else:
                break
        a[j+1] = key
        f.append((list(a), j+1, i, C, S, f'place key at {j+1}'))
    f.append((list(a), -1, -1, C, S, 'sorted')); return f

def selection_frames(a):
    a = list(a); f = [(list(a), -1, -1, 0, 0, 'start')]; C = S = 0
    for i in range(len(a)):
        m = i
        for j in range(i+1, len(a)):
            C += 1
            if a[j] < a[m]: m = j
            f.append((list(a), m, j, C, S, f'scan min in [{i}:], min@{m}'))
        if m != i:
            a[i], a[m] = a[m], a[i]; S += 1
        f.append((list(a), i, m, C, S, f'place min at {i}'))
    return f

BUILDERS = {'bubble': bubble_frames, 'insertion': insertion_frames, 'selection': selection_frames}

algo_s = widgets.Dropdown(options=list(BUILDERS), value='bubble', description='algorithm')
arr_s  = widgets.Dropdown(options=['random','sorted','reversed'], value='random', description='input')
n_s    = widgets.IntSlider(value=12, min=5, max=20, description='n')
seed_s = widgets.IntSlider(value=1, min=0, max=20, description='seed')
sort_area = widgets.Output()

def relaunch_sort(*_):
    rng = np.random.default_rng(seed_s.value)
    if arr_s.value == 'random':   a = rng.permutation(n_s.value)
    elif arr_s.value == 'sorted': a = np.arange(n_s.value)
    else:                         a = np.arange(n_s.value)[::-1]
    frames = BUILDERS[algo_s.value](a); vmax = max(a)
    def draw(k):
        arr, h1, h2, C, S, note = frames[k]
        colors = ['lightsteelblue']*len(arr)
        for h in (h1, h2):
            if 0 <= h < len(arr): colors[h] = 'tomato'
        fig, ax = plt.subplots()
        ax.bar(range(len(arr)), arr, color=colors); ax.set_ylim(0, vmax*1.1)
        ax.set_title(f'{algo_s.value} . step {k}/{len(frames)-1} . C={C} S={S} . {note}')
        ax.set_xlabel('index'); ax.set_ylabel('value'); plt.show()
    sort_area.clear_output(wait=True)
    with sort_area: make_player(len(frames), draw)

for w in (algo_s, arr_s, n_s, seed_s): w.observe(relaunch_sort, 'value')
display(widgets.HBox([algo_s, arr_s]), widgets.HBox([n_s, seed_s]), sort_area)
relaunch_sort()

Output()

## Merge Sort — Watch the Merges Combine

Merge sort treats the array as singletons and merges sorted runs of doubling width. Recording the state after each completed merge (highlighted span) shows order emerging bottom-up, level by level, in $O(n\log n)$.

$$ T(n) = 2\,T\!\left(\tfrac{n}{2}\right) + O(n) = O(n \log n). $$

In [6]:
def merge_frames(a):
    a = list(a); frames = [(list(a), set(), 'start: singletons')]; n = len(a); width = 1
    while width < n:
        for lo in range(0, n, 2*width):
            mid = min(lo+width, n); hi = min(lo+2*width, n)
            left, right = a[lo:mid], a[mid:hi]; i = j = 0; merged = []
            while i < len(left) and j < len(right):
                if left[i] <= right[j]: merged.append(left[i]); i += 1
                else: merged.append(right[j]); j += 1
            merged += left[i:] + right[j:]; a[lo:hi] = merged
            frames.append((list(a), set(range(lo, hi)), f'merge width {width} at [{lo}:{hi}]'))
        width *= 2
    frames.append((list(a), set(range(n)), 'sorted')); return frames

n_s = widgets.IntSlider(value=16, min=4, max=32, step=2, description='n')
seed_s = widgets.IntSlider(value=2, min=0, max=20, description='seed')
merge_area = widgets.Output()

def relaunch_merge(*_):
    rng = np.random.default_rng(seed_s.value)
    a = rng.permutation(n_s.value); frames = merge_frames(a); vmax = max(a)
    def draw(k):
        arr, hot, note = frames[k]
        colors = ['seagreen' if i in hot else 'lightsteelblue' for i in range(len(arr))]
        fig, ax = plt.subplots()
        ax.bar(range(len(arr)), arr, color=colors); ax.set_ylim(0, vmax*1.1)
        ax.set_title(f'merge sort . step {k}/{len(frames)-1} . {note}')
        ax.set_xlabel('index'); ax.set_ylabel('value'); plt.show()
    merge_area.clear_output(wait=True)
    with merge_area: make_player(len(frames), draw)

n_s.observe(relaunch_merge, 'value'); seed_s.observe(relaunch_merge, 'value')
display(widgets.HBox([n_s, seed_s]), merge_area)
relaunch_merge()

Output()

## Quick Sort — Watch Partitioning (and Its Worst Case)

Quick sort moves a pivot (red) to its final place while elements smaller than it accumulate in the green region, then recurses left and right. On already-sorted input with a last-element pivot the partitions stay maximally unbalanced — many more steps — which *is* the $O(n^2)$ degeneration. Switch the pivot to random to recover $O(n\log n)$.

$$ T_{\text{best}}=O(n\log n), \qquad T_{\text{worst}}=O(n^2). $$

In [7]:
def quick_frames(a, pivot_mode, rng):
    a = list(a); frames = [(list(a), -1, set(), 'start')]
    def qs(lo, hi):
        if lo >= hi: return
        p = hi if pivot_mode == 'last' else rng.integers(lo, hi+1)
        a[p], a[hi] = a[hi], a[p]; pivot = a[hi]; i = lo
        for j in range(lo, hi):
            if a[j] < pivot:
                a[i], a[j] = a[j], a[i]; i += 1
                frames.append((list(a), hi, set(range(lo, i)), f'partition [{lo}:{hi}] pivot={pivot}'))
        a[i], a[hi] = a[hi], a[i]
        frames.append((list(a), i, set(range(lo, hi+1)), f'pivot {pivot} placed at {i}'))
        qs(lo, i-1); qs(i+1, hi)
    qs(0, len(a)-1)
    frames.append((list(a), -1, set(range(len(a))), 'sorted')); return frames

arr_s = widgets.Dropdown(options=['random','sorted','reversed'], value='sorted', description='input')
piv_s = widgets.Dropdown(options=['last','random'], value='last', description='pivot')
n_s   = widgets.IntSlider(value=12, min=5, max=18, description='n')
seed_s= widgets.IntSlider(value=1, min=0, max=20, description='seed')
quick_area = widgets.Output()

def relaunch_quick(*_):
    rng = np.random.default_rng(seed_s.value)
    if arr_s.value == 'random':   a = rng.permutation(n_s.value)
    elif arr_s.value == 'sorted': a = np.arange(n_s.value)
    else:                         a = np.arange(n_s.value)[::-1]
    frames = quick_frames(a, piv_s.value, rng); vmax = max(a)
    def draw(k):
        arr, piv, region, note = frames[k]
        colors = ['lightsteelblue']*len(arr)
        for i in region: colors[i] = 'palegreen'
        if 0 <= piv < len(arr): colors[piv] = 'tomato'
        fig, ax = plt.subplots()
        ax.bar(range(len(arr)), arr, color=colors); ax.set_ylim(0, vmax*1.1)
        ax.set_title(f'quick sort . step {k}/{len(frames)-1} . {note}')
        ax.set_xlabel('index'); ax.set_ylabel('value'); plt.show()
    quick_area.clear_output(wait=True)
    with quick_area: make_player(len(frames), draw)

for w in (arr_s, piv_s, n_s, seed_s): w.observe(relaunch_quick, 'value')
display(widgets.HBox([arr_s, piv_s]), widgets.HBox([n_s, seed_s]), quick_area)
relaunch_quick()

Output()

## Kadane's Algorithm — Watch the Running Sum

Kadane scans once, extending the current subarray or resetting when starting fresh at $x_i$ beats continuing. Stepping through shows the cursor (red), the best window so far (green), and the running value `cur` traced in the lower panel — including the reset moments where `cur` drops back to a single element.

$$ \text{cur}_i = \max(x_i,\ \text{cur}_{i-1} + x_i), \qquad \text{best} = \max_i \text{cur}_i. $$

In [ ]:
def kadane_frames(x):
    cur = best = x[0]; start = 0; lo = hi = 0
    frames = [(0, int(cur), int(best), 0, 0, 'init')]
    for i in range(1, len(x)):
        if x[i] > cur + x[i]:
            cur = x[i]; start = i; note = f'reset at {i}'
        else:
            cur = cur + x[i]; note = f'extend to {i}'
        if cur > best:
            best = cur; lo, hi = start, i
        frames.append((i, int(cur), int(best), lo, hi, note))
    return frames

n_s = widgets.IntSlider(value=18, min=5, max=30, description='n')
seed_s = widgets.IntSlider(value=3, min=0, max=30, description='seed')
kad_area = widgets.Output()

def relaunch_kad(*_):
    rng = np.random.default_rng(seed_s.value)
    x = rng.integers(-9, 10, n_s.value); frames = kadane_frames(x); n = len(x)
    def draw(k):
        i, cur, best, lo, hi, note = frames[k]
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
        colors = []
        for j in range(n):
            if j == i: colors.append('tomato')
            elif lo <= j <= hi: colors.append('seagreen')
            else: colors.append('lightsteelblue')
        ax1.bar(range(n), x, color=colors)
        ax1.set_title(f'step {k}/{len(frames)-1} . {note} . cur={cur} best={best} (window {lo}..{hi})')
        ax1.set_ylabel('value')
        curve = [f[1] for f in frames[:k+1]]
        ax2.plot(range(k+1), curve, 'o-', ms=3, color='darkorange')
        ax2.axhline(0, color='k', lw=0.8); ax2.set_xlim(0, n-1)
        ax2.set_ylabel('cur'); ax2.set_xlabel('index'); plt.tight_layout(); plt.show()
    kad_area.clear_output(wait=True)
    with kad_area: make_player(len(frames), draw)

n_s.observe(relaunch_kad, 'value'); seed_s.observe(relaunch_kad, 'value')
display(widgets.HBox([n_s, seed_s]), kad_area)
relaunch_kad()

Output()